# This notebook serves as walkthrough for planning an experiment for creation through the OT2.
### The following modules are used and should be in the same directory as this notebook: 
* **CreateSamples** is responsible for calculating sample information, which includes component weight fractions and stock volumes
* **OT2Commands** is responsible for setting up information to be interpretted and executed by opentrons **BASED ON THE LATEST API 2.3 and above**
* **OT2Graphing** contains graphing tools to help visualize and explore parameter spaces.

In [1]:
import os
import warnings
os.getcwd()

'/var/lib/jupyter/notebooks/Huat'

In [3]:
#os.system("systemctl start opentrons-robot-server")

In [4]:
from Plan import CreateSamples
from Prepare import OT2Commands as ALH
#from Prepare import OT2Graphing as ographing
from opentrons import simulate, execute, protocol_api
from Prepare import OT2Thermocycler as TC
import importlib
import pandas as pd
from datetime import datetime
from pytz import timezone 

_**Note**_  : If your file is not in the same folder, you can specify its path by using '\' and the folder names. 

* to go up a folder use this : '../'
* to use the same folder: './' 

Be consistent with the slashes you use , either all: '\' or '/' 

In [5]:
path = r"Crystallization_Protocol_Wellplate.csv"
chem_path = r"Chemical Database.csv"
plan = CreateSamples.get_experiment_plan(path, chem_path)

In [ ]:
stock_volumes = CreateSamples.concentration_from_csv('Volumes/Experiment_40_Samples_Part_1')
stock_volumes = stock_volumes.loc[:, ~stock_volumes.columns.str.contains('^Unnamed')]
#stock_volumes['DNALinker-stock'].sum(axis=0)
stock_volumes.astype(int)

,DNA-Bridge (6 uM)-stock
0,7
1,7
2,7
3,7
4,10
5,10
6,10
7,10
8,13
9,13


In [7]:
stock_volumes.sum(axis=0)

DNA-Bridge (6 uM)-stock    188.0
dtype: float64

## Simulate

Run the following cells to simulate the robot protocol. 
If you look at the last cell in this section, you can see the actual steps the robot will follow to make the samples.

In [8]:
# Simulating 
labware_dir_path = r"Custom Labware"
custom_labware_dict = ALH.custom_labware_dict(labware_dir_path)
protocol = simulate.get_protocol_api('2.8', extra_labware = custom_labware_dict) #

/data/robot_settings.json not found. Loading defaults
/data/deck_calibration.json not found. Loading defaults


In [9]:
loaded_dict = ALH.loading_labware(protocol, plan)
max_source_vol = 17350
stock_position_info = ALH.stock_well_ranges(stock_volumes, loaded_dict, max_source_vol) 

# let's print the stock position info and make sure everything is in the right postion. 
# You can also double-check that you have enought stock to make all your samples
# (i.e. only a single range of wells is provided for each stock)

dest offset loaded
stock offset not loaded


In [10]:
stock_position_info

{'DNA-Bridge (6 uM)-stock': {'Ranges': [[0, 16]],
  'Stock Wells': [A1 of 20mLscintillation 12 Well Plate 18000 µL on 1]}}

In [11]:
Start_pos = 0

In [12]:
directions = ALH.create_sample_making_directions(stock_volumes, stock_position_info, loaded_dict, start_position=Start_pos)
ALH.pipette_volumes_component_wise(protocol, directions, loaded_dict)
# The above command will fill the wells component wise- this means that all the first stock will be dispendes in the wells that require it. 
# If you want to fill the wells individually use the following line instead:
# ALH.pipette_volumes_sample_wise(protocol,directions, loaded_dict)

Picking up tip from A1 of Opentrons OT-2 96 Tip Rack 20 µL on 4
Aspirating 7.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Air gap
Aspirating 13.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Dispensing 7.0 uL into A1 of USAXS 48 Well Plate 500 µL on 6 at 700.0 uL/sec
Blowing out at A1 of USAXS 48 Well Plate 500 µL on 6
Aspirating 7.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Air gap
Aspirating 13.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Dispensing 7.0 uL into A2 of USAXS 48 Well Plate 500 µL on 6 at 700.0 uL/sec
Blowing out at A2 of USAXS 48 Well Plate 500 µL on 6
Aspirating 7.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Air gap
Aspirating 13.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Dispensing 7.0 uL into A3 of USAXS 48 Well Plate 500 µL on 6 at 700.0 uL/sec
Blowing out at A3 of USAXS

In [13]:
#os.system("systemctl stop opentrons-robot-server")

## Execute


Run the cells below to actually execute the protocol using the robot.

Make sure that all stock solution are **not capped** and that the labware is in the correct position. 

In [14]:
#os.system("systemctl start opentrons-robot-server")

In [15]:
# Executing 
protocol = execute.get_protocol_api('2.8', extra_labware = custom_labware_dict)

/data/robot_settings.json not found. Loading defaults
Failed to initialize character device, will not be able to control gpios (lights, button, smoothiekill, smoothie reset). Only one connection can be made to the gpios at a time. If you need to control gpios, first stop the robot server with systemctl stop opentrons-robot-server. Until you restart the server with systemctl start opentrons-robot-server, you will be unable to control the robot using the Opentrons app.
/data/deck_calibration.json not found. Loading defaults


In [16]:
loaded_dict = ALH.loading_labware(protocol, plan)
stock_position_info = ALH.stock_well_ranges(stock_volumes, loaded_dict, max_source_vol) 
stock_position_info

dest offset loaded
stock offset not loaded


{'DNA-Bridge (6 uM)-stock': {'Ranges': [[0, 16]],
  'Stock Wells': [A1 of 20mLscintillation 12 Well Plate 18000 µL on 1]}}

In [17]:
directions = ALH.create_sample_making_directions(stock_volumes, stock_position_info, loaded_dict, start_position=Start_pos)
directions

{0: {'DNA-Bridge (6 uM)-stock': {'Stock Position': A1 of 20mLscintillation 12 Well Plate 18000 µL on 1,
   'Destination Well Position': A1 of USAXS 48 Well Plate 500 µL on 6,
   'Stock Volume': 7.0}},
 1: {'DNA-Bridge (6 uM)-stock': {'Stock Position': A1 of 20mLscintillation 12 Well Plate 18000 µL on 1,
   'Destination Well Position': A2 of USAXS 48 Well Plate 500 µL on 6,
   'Stock Volume': 7.0}},
 2: {'DNA-Bridge (6 uM)-stock': {'Stock Position': A1 of 20mLscintillation 12 Well Plate 18000 µL on 1,
   'Destination Well Position': A3 of USAXS 48 Well Plate 500 µL on 6,
   'Stock Volume': 7.0}},
 3: {'DNA-Bridge (6 uM)-stock': {'Stock Position': A1 of 20mLscintillation 12 Well Plate 18000 µL on 1,
   'Destination Well Position': A4 of USAXS 48 Well Plate 500 µL on 6,
   'Stock Volume': 7.0}},
 4: {'DNA-Bridge (6 uM)-stock': {'Stock Position': A1 of 20mLscintillation 12 Well Plate 18000 µL on 1,
   'Destination Well Position': A5 of USAXS 48 Well Plate 500 µL on 6,
   'Stock Volume': 10

### the following command will actully start the experiment. Once again make sure that everything is where it is supposed to be!

In [18]:
#ALH.thermocycler_open_lid(protocol, loaded_dict)

In [19]:
ALH.pipette_volumes_component_wise(protocol, directions, loaded_dict)

Picking up tip from A1 of Opentrons OT-2 96 Tip Rack 20 µL on 4
Aspirating 7.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Air gap
Aspirating 13.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Dispensing 7.0 uL into A1 of USAXS 48 Well Plate 500 µL on 6 at 700.0 uL/sec
Blowing out at A1 of USAXS 48 Well Plate 500 µL on 6
Aspirating 7.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Air gap
Aspirating 13.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Dispensing 7.0 uL into A2 of USAXS 48 Well Plate 500 µL on 6 at 700.0 uL/sec
Blowing out at A2 of USAXS 48 Well Plate 500 µL on 6
Aspirating 7.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Air gap
Aspirating 13.0 uL from A1 of 20mLscintillation 12 Well Plate 18000 µL on 1 at 100.0 uL/sec
Dispensing 7.0 uL into A3 of USAXS 48 Well Plate 500 µL on 6 at 700.0 uL/sec
Blowing out at A3 of USAXS

In [58]:
#ALH.thermocycler_close_lid(protocol, loaded_dict)

In [ ]:
Pacific = timezone('US/Pacific')
p_time = datetime.now(Pacific)
print(p_time.strftime('%H:%M:%S'))

In [ ]:
import numpy as np
temp = np.linspace(25,25,451).reshape(-1,1)
time = np.linspace(10,10,451).reshape(-1,1)
temp_time = np.hstack((temp, time))

max_volume = 200
ALH.thermocycler_temperature(protocol, loaded_dict, temp_time, max_volume)

In [ ]:
Pacific = timezone('US/Pacific')
p_time = datetime.now(Pacific)
print(p_time.strftime('%H:%M:%S'))

In [32]:
os.system("systemctl start opentrons-robot-server")

0

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/serial/serialposix.py", line 621, in write
OSError: [Errno 19] No such device

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/thermocycler.py", line 651, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/thermocycler.py", line 656, in read_lid_status
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/thermocycler/driver.py", line 193, in get_lid_status
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 135, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 170, in send_data
  File "/